# 🎬 VSL Forge — MECProAI
## Geração de Vídeos dos Cursos + Criativos de Campanha + API ngrok

### ⚠️ ANTES DE RODAR:
1. Vá em **Runtime → Change runtime type → T4 GPU**
2. Execute as células **em ordem**
3. Na Célula 3, faça upload dos arquivos necessários

### ⏱️ Tempo estimado:
| Modo | Por vídeo | 8 cursos completos |
|------|-----------|--------------------|
| `--draft` (teste, 480p) | ~2-3 min | ~16-24 min |
| Final (qualidade, 1080p) | ~5-10 min | ~40-80 min |

### 📋 Fluxo completo:
```
Célula 1 → Instalar dependências
Célula 2 → Configurar chaves (já preenchidas)
Célula 3 → Upload dos arquivos Python + JSON
Célula 4 → Listar cursos disponíveis
Célula 5 → [TESTE] Gerar 1 cena rápida para validar
Célula 6 → Gerar todos os cursos (ou um específico)
Célula 7 → Salvar vídeos no Google Drive
Célula 8 → [OPCIONAL] Iniciar API + ngrok para MECProAI
Célula 9 → [OPCIONAL] Verificar API + listar formatos
Célula 10 → [OPCIONAL] Testar geração via API
Célula 11 → Manter Colab vivo (anti-timeout)
```

## 📦 CÉLULA 1 — Instalar dependências

In [ ]:
import subprocess, sys

# Sistema
print('📦 Instalando dependências do sistema...')
subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], capture_output=True)

# Python
print('🐍 Instalando dependências Python...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'fastapi', 'uvicorn', 'python-multipart', 'aiofiles',
    'pyngrok', 'nest-asyncio', 'requests', 'Pillow',
    'tqdm', 'elevenlabs', 'huggingface_hub'
], capture_output=True)

# Verificações
ffmpeg = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print('\n✅ FFmpeg      :', 'OK' if ffmpeg.returncode == 0 else '❌ FALHOU')
print('✅ Python libs : OK')
print('\n🟢 Pronto! Execute a próxima célula.')

## 🔑 CÉLULA 2 — Configurar API Keys

In [ ]:
import os

# ══════════════════════════════════════════════
# KEYS CONFIGURADAS — altere se necessário
# ══════════════════════════════════════════════
ELEVENLABS_API_KEY  = "sk_d524c8cb9bdbe0e620dea85ff383eae12ce9eba2d7ab8a3f"
HUGGINGFACE_API_KEY = " "
NGROK_AUTH_TOKEN    = "3Bi8Slx2ypO1ep9gDaUvdFfwO5h_2bikKxPRea46yeGetHzV9"
NGROK_DOMAIN        = "arcane-embryologic-tiffiny.ngrok-free.dev"  # domínio fixo
VSL_SECRET_KEY      = "mecpro-vsl-2026"
# ══════════════════════════════════════════════

os.environ['ELEVENLABS_API_KEY']  = ELEVENLABS_API_KEY
os.environ['HUGGINGFACE_API_KEY'] = HUGGINGFACE_API_KEY
os.environ['VSL_SECRET_KEY']      = VSL_SECRET_KEY
os.environ['OUTPUT_DIR']          = '/content/vsl_outputs'

os.makedirs('/content/vsl_outputs', exist_ok=True)
os.makedirs('/content/academy_output', exist_ok=True)

print('✅ ElevenLabs  :', ELEVENLABS_API_KEY[:12] + '...')
print('✅ HuggingFace :', HUGGINGFACE_API_KEY[:12] + '...')
print('✅ ngrok token :', NGROK_AUTH_TOKEN[:12] + '...')
print('✅ ngrok domain:', NGROK_DOMAIN)
print('✅ VSL Key     :', VSL_SECRET_KEY)
print('\n🟢 Chaves configuradas!')

## 📥 CÉLULA 3 — Upload dos arquivos
Faça upload dos seguintes arquivos do seu computador:

In [ ]:
from google.colab import files
import os

print('📤 Selecione os arquivos para upload:')
print('   1. vsl_forge_v2.py      ← obrigatório')
print('   2. vsl_service.py       ← obrigatório (para API)')
print('   3. academy_cursos.json  ← obrigatório (cursos)')
print('   4. split_cursos.py      ← opcional')
print()

uploaded = files.upload()

for filename, data in uploaded.items():
    dest = f'/content/{filename}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  ✅ {filename} ({len(data)//1024} KB)')

print()
needed = ['vsl_forge_v2.py', 'vsl_service.py', 'academy_cursos.json']
all_ok = True
for fname in needed:
    exists = os.path.exists(f'/content/{fname}')
    print(f'  {"✅" if exists else "❌ FALTANDO"} {fname}')
    if not exists: all_ok = False

print()
if all_ok:
    print('🟢 Todos os arquivos carregados! Pode continuar.')
else:
    print('🔴 Faça upload dos arquivos faltando antes de continuar.')

## 📚 CÉLULA 4 — Listar cursos disponíveis

In [ ]:
import json

with open('/content/academy_cursos.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

cursos = data['cursos']
print(f'📚 {len(cursos)} cursos encontrados:\n')
total_cenas = 0
for i, c in enumerate(cursos, 1):
    n = len(c['scenes'])
    total_cenas += n
    print(f'  {i}. {c["slug"]:45s} ({n} cenas)')

print(f'\n  Total de cenas : {total_cenas}')
print(f'  Tempo draft    : ~{total_cenas * 2} min')
print(f'  Tempo final    : ~{total_cenas * 7} min')

## 🧪 CÉLULA 5 — Teste rápido (1 cena)
**Rode isso primeiro** para confirmar que tudo está funcionando antes de gerar os 8 cursos.

In [ ]:
import json, subprocess, sys

test_project = {
    'title': 'Teste MECProAI',
    'output_name': 'teste_mecpro',
    'tts_engine': 'elevenlabs',
    'voice': 'hpp4J3VqNfWAUOO0d1Us',
    'add_subtitles': True,
    'color_grade': 'warm_film',
    'vignette': True,
    'crossfade_duration': 0.8,
    'normalize_audio': True,
    'master_music_gain_db': -20,
    'scenes': [{
        'id': 'hook',
        'narration': 'Descubra como dobrar seus resultados com inteligência artificial.',
        'prompt': 'Professional Brazilian entrepreneur smiling at laptop, modern office, cinematic lighting',
        'motion': 'push_in',
        'xfade': 'dissolve',
        'xfade_duration': 0.8,
        'cinematic_bars': False,
        'grain': 0.06,
        'color_grade': 'warm_film',
        'audio_fade_in': 0.05,
        'audio_fade_out': 0.3,
    }]
}

with open('/content/test_project.json', 'w') as f:
    json.dump(test_project, f, indent=2)

print('▶️  Rodando teste em modo draft (1 cena, ~2 min)...')
result = subprocess.run(
    [sys.executable, '/content/vsl_forge_v2.py',
     '--config', '/content/test_project.json', '--draft'],
    text=True
)
print('\nCódigo de saída:', result.returncode)
if result.returncode == 0:
    print('\n✅ Teste passou! VSL Forge funcionando.')
    print('   Pode prosseguir para a Célula 6 para gerar os cursos.')
else:
    print('\n❌ Teste falhou. Verifique os logs acima antes de continuar.')

## 🎬 CÉLULA 6 — Gerar vídeos dos cursos

**Configurações:**
- `MODO_DRAFT = True` → 480p, rápido, para testar
- `MODO_DRAFT = False` → 1080p, qualidade final
- `CURSO_ESPECIFICO = None` → gera **todos** os 8 cursos
- `CURSO_ESPECIFICO = 'campanha-zero-mecpro'` → gera só um

💡 Use `--resume`: se o Colab cair, retoma de onde parou automaticamente.

In [ ]:
import json, subprocess, sys, os
from pathlib import Path

# ══════════════════════════════════════════════
# CONFIGURAÇÃO — altere aqui
MODO_DRAFT       = True   # True = draft rápido | False = qualidade final
CURSO_ESPECIFICO = None   # None = todos | ex: 'campanha-zero-mecpro'
# ══════════════════════════════════════════════

with open('/content/academy_cursos.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

cursos = data['cursos']
if CURSO_ESPECIFICO:
    cursos = [c for c in cursos if c['slug'] == CURSO_ESPECIFICO]
    if not cursos:
        print(f'❌ Curso "{CURSO_ESPECIFICO}" não encontrado!')
        print('Disponíveis:', [c['slug'] for c in data['cursos']])
        raise SystemExit(1)

# Salvar JSONs individuais
output_dir = Path('/content/academy_output')
output_dir.mkdir(exist_ok=True)
for c in cursos:
    with open(output_dir / f"{c['slug']}.json", 'w', encoding='utf-8') as f:
        json.dump(c, f, indent=2, ensure_ascii=False)

modo_str = '⚡ DRAFT (480p)' if MODO_DRAFT else '🎬 FINAL (1080p)'
print(f'Modo   : {modo_str}')
print(f'Cursos : {len(cursos)}')
print(f'Resume : ✅ (retoma de onde parou se cair)\n')

resultados = []
for i, curso in enumerate(cursos, 1):
    slug = curso['slug']
    config = f'/content/academy_output/{slug}.json'
    print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
    print(f'[{i}/{len(cursos)}] 🎬 {slug}')
    print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

    cmd = [sys.executable, '/content/vsl_forge_v2.py',
           '--config', config, '--resume']
    if MODO_DRAFT:
        cmd.append('--draft')

    result = subprocess.run(cmd, text=True)
    ok = result.returncode == 0
    resultados.append((slug, ok))
    print(f'  → {"✅ OK" if ok else "❌ FALHOU"}: {slug}\n')

print('\n' + '═'*50)
print('RESUMO FINAL')
print('═'*50)
for slug, ok in resultados:
    print(f'  {"✅" if ok else "❌"} {slug}')
sucessos = sum(1 for _, ok in resultados if ok)
print(f'\n  {sucessos}/{len(resultados)} cursos gerados com sucesso')
print(f'\n📁 Vídeos em: /content/vsl_build/')

## 💾 CÉLULA 7 — Salvar vídeos no Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

DRIVE_DEST = Path('/content/drive/MyDrive/MECProAI/VSL_Videos')
DRIVE_DEST.mkdir(parents=True, exist_ok=True)

build_dir = Path('/content/vsl_build')
copiados = 0

if not build_dir.exists():
    print('⚠️  Nenhum vídeo gerado ainda. Rode a Célula 6 primeiro.')
else:
    for mp4 in sorted(build_dir.rglob('final/*.mp4')):
        dest = DRIVE_DEST / mp4.name
        shutil.copy2(mp4, dest)
        size_mb = mp4.stat().st_size / 1024**2
        print(f'  ✅ {mp4.name} ({size_mb:.1f} MB) → Drive')
        copiados += 1

    print(f'\n🎉 {copiados} vídeo(s) salvos em:')
    print(f'   📁 Google Drive → MECProAI/VSL_Videos')

## 🌐 CÉLULA 8 — Iniciar API + ngrok
**Opcional** — só precisa se quiser que o MECProAI (Render) chame a geração automaticamente.

Para gerar vídeos manualmente, as células 5, 6 e 7 já são suficientes.

**URL fixa no Render:**
```
VSL_SERVICE_URL = https://arcane-embryologic-tiffiny.ngrok-free.dev
VSL_SECRET_KEY  = mecpro-vsl-2026
```

In [ ]:
import nest_asyncio, uvicorn, threading, time, sys, importlib.util
from pyngrok import ngrok, conf

nest_asyncio.apply()
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Carregar o vsl_service
sys.path.insert(0, '/content')
spec = importlib.util.spec_from_file_location('vsl_service', '/content/vsl_service.py')
vsl_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vsl_module)
app = vsl_module.app

# Iniciar FastAPI em thread separada
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)

# Túnel ngrok com domínio FIXO
tunnel = ngrok.connect(8000, 'http', domain=NGROK_DOMAIN)
public_url = f'https://{NGROK_DOMAIN}'

print('=' * 60)
print('🚀 VSL Service online!')
print('=' * 60)
print(f'\n🌐 URL pública : {public_url}')
print(f'🔍 Health check: {public_url}/health')
print(f'\n📋 Variáveis para o Render:')
print(f'   VSL_SERVICE_URL = {public_url}')
print(f'   VSL_SECRET_KEY  = {VSL_SECRET_KEY}')
print(f'\n✅ Domínio FIXO — configure no Render uma única vez!')

## 🔍 CÉLULA 9 — Verificar API + listar formatos

In [ ]:
import requests

headers = {'Authorization': f'Bearer {VSL_SECRET_KEY}'}

# Health check
r = requests.get(f'{public_url}/health', headers=headers)
d = r.json()
print('🔍 Status do serviço:')
print(f'  FFmpeg      : {"✅" if d.get("ffmpeg") else "❌"}')
print(f'  HuggingFace : {"✅" if d.get("huggingface") else "❌"}')
print(f'  ElevenLabs  : {"✅" if d.get("elevenlabs") else "❌"}')
print(f'  Jobs ativos : {d.get("jobs_active", 0)}')

# Formatos disponíveis
r2 = requests.get(f'{public_url}/formats', headers=headers)
print('\n📋 Formatos disponíveis:')
for k, v in r2.json().items():
    print(f'  {k:20s} {v["label"]:30s} {v["resolution"]}')

## 🎬 CÉLULA 10 — Testar geração via API
Gera 1 vídeo de teste chamando a API pelo ngrok (simula o que o MECProAI faz).

In [ ]:
import requests, time

headers = {'Authorization': f'Bearer {VSL_SECRET_KEY}', 'Content-Type': 'application/json'}

payload = {
    'title': 'Teste via API — MECProAI',
    'format': 'reels_9_16',
    'project_name': 'teste_api',
    'draft': True,
    'scenes': [{
        'id': 'hook',
        'narration': 'MECProAI transforma suas campanhas com inteligência artificial.',
        'prompt': 'Brazilian professional woman smiling at smartphone, modern city background, cinematic',
        'motion': 'push_in',
        'xfade': 'dissolve',
        'xfade_duration': 0.8,
        'cinematic_bars': False,
        'grain': 0.06,
        'color_grade': 'warm_film',
        'audio_fade_in': 0.05,
        'audio_fade_out': 0.3,
    }]
}

# Iniciar job
r = requests.post(f'{public_url}/generate', json=payload, headers=headers)
job = r.json()
job_id = job['job_id']
print(f'✅ Job iniciado  : {job_id}')
print(f'⏱️  Estimativa   : {job["estimated_min"]} minutos')

# Polling de status
print('\nAguardando geração...')
while True:
    time.sleep(10)
    r2 = requests.get(f'{public_url}/status/{job_id}', headers=headers)
    status = r2.json()
    print(f'  [{status["progress"]:3d}%] {status["message"]}')
    if status['status'] in ['done', 'error']:
        break

if status['status'] == 'done':
    print(f'\n🎉 Vídeo pronto!')
    print(f'   URL      : {public_url}{status["video_url"]}')
    print(f'   Duração  : {status["duration_s"]}s')
    print(f'   Tamanho  : {status["size_mb"]} MB')
else:
    print(f'\n❌ Erro: {status["error"]}')

## 🔄 CÉLULA 11 — Manter Colab vivo (anti-timeout)
Execute esta célula para o Colab não desconectar enquanto gera vídeos ou mantém a API online.
Pare com o botão **■** quando terminar.

In [ ]:
import time, requests
from pathlib import Path

print('🔄 Anti-timeout ativo. Pressione ■ para parar.\n')

minutos = 0
while True:
    time.sleep(60)
    minutos += 1

    # Contar vídeos gerados
    build = Path('/content/vsl_build')
    videos = list(build.rglob('final/*.mp4')) if build.exists() else []

    # Verificar API se estiver rodando
    api_status = ''
    try:
        r = requests.get(
            f'https://{NGROK_DOMAIN}/health',
            headers={'Authorization': f'Bearer {VSL_SECRET_KEY}'},
            timeout=5
        )
        jobs = r.json().get('jobs_active', 0)
        api_status = f' | API ✅ | Jobs: {jobs}'
    except:
        pass

    print(f'  ⏱️  {minutos} min — {len(videos)} vídeo(s) gerado(s){api_status}', flush=True)